# Demo: using the `insurance_cross_sell` package

The notebook only *calls* the package (no model code lives here). Run `poetry run insurance download` first, or point `DATA_PATH` to your copy of `train.csv`.

In [ ]:
from insurance_cross_sell import config
from insurance_cross_sell.bundle import ModelBundle
from insurance_cross_sell.evaluate import feature_importance
from insurance_cross_sell.train import fit_and_evaluate, prepare_split

DATA_PATH = config.DATA_PATH
NROWS = None  # e.g. 1_000_000 to use only the head of the file

## 1. Load a stratified sample and split
Chunked reading keeps memory low; the split happens before any fitting.

In [ ]:
split = prepare_split(DATA_PATH, sample_frac=config.SAMPLE_FRAC, nrows=NROWS)
split.X_train.head()

## 2. Fit the final model with the stored best hyperparameters
No search is run: `config.BEST_PARAMS` holds the results of the original Optuna/Grid runs.

In [ ]:
bundle = fit_and_evaluate('lightgbm', split)
bundle.metrics

## 3. Feature importance (names aligned with the transformed columns)

In [ ]:
importance = feature_importance(bundle.pipeline)
importance.head(10)

In [ ]:
importance.head(10).set_index('feature')['importance'].sort_values().plot.barh(figsize=(7, 4));

## 4. Save, reload and predict on raw rows
The saved bundle includes the pipeline and threshold; input is validated against the schema.

In [ ]:
path = bundle.save(config.ARTIFACTS_DIR / 'demo' / 'model.joblib')
loaded = ModelBundle.load(path)
loaded.predict(split.X_test.head(5))